# 03 — Main run (sim + price + export)
Full chain once per scenario: `gen()` → `simulate()` → price N regimes → scorecards → `shared/` → Excel export. All figures, diagnostics, Monte Carlo and verdicts live in `04_analysis.ipynb`.

Run top to bottom. Only touch the config cell.

In [ ]:
import copy, os, sys
ROOT = os.path.abspath('')
if ROOT not in sys.path: sys.path.insert(0, ROOT)

import pandas as pd
from voltvision import CFG, SCEN, N_YEARS, REGIMES
from voltvision import gen, simulate, price_many, summary, lr_by
from voltvision import ensure_core_methods
from voltvision import io
ensure_core_methods()  # load CALC cells from 02a/02b/02c


## Config (only cell you need to touch)

In [ ]:
VEH = "ALL"   # "ICE" | "EV" | "MIX" | "ALL"
QUICK = False  # True -> n=1000, 2 years (smoke test)

cfg = copy.deepcopy(CFG)
if QUICK:
    cfg['n'] = 1000
scens = [VEH] if VEH in SCEN else ["ICE", "EV", "MIX"]
REGIMES_RUN = list(REGIMES)  # add registered names here for N-regime runs
nyears = 2 if QUICK else N_YEARS
print(scens, '| n =', cfg['n'], '| years =', nyears)


## Results — one scorecard per scenario (all regimes side by side)

In [ ]:
ALL = {}
for s in scens:
    vp = SCEN[s]
    d0 = gen(cfg, vp, cfg['seed'])
    book = simulate(d0, cfg, vp, seed=cfg['seed'], n_years=nyears, verbose=True)
    io.save_sim(s, book)
    books = price_many(book, cfg, REGIMES_RUN)
    for m, b in books.items():
        io.save_priced(s, m, b)
    ALL[s] = (book, books)
    print(f"\n=== {s} {vp} ===")
    display(summary(books))
    print("LR by coverage (rows=coverage, cols=regime):")
    display(pd.DataFrame({m: lr_by(b, 'COVERAGE_TYPE').round(1) for m, b in books.items()}))
print("\nLR by vehicle (MIX book, all regimes):")
if "MIX" in ALL:
    display(pd.DataFrame({m: lr_by(b, 'VEHICLE_TYPE').round(1) for m, b in ALL["MIX"][1].items()}))


## Excel export (Simulation Data + one premium sheet per regime)
Nothing but the thin sheets here — heavy analysis stays in `04`.

In [ ]:
scen_to_export = "MIX" if "MIX" in ALL else list(ALL.keys())[0]
raw_book, books = ALL[scen_to_export]
file_name = f"VoltVision_Motor_Simulation_{scen_to_export}.xlsx"
with pd.ExcelWriter(file_name, engine='openpyxl') as writer:
    raw_book.to_excel(writer, sheet_name='Simulation Data', index=False)
    for m, b in books.items():
        b[['POLID', 'FINAL_PREMIUM_SST']].to_excel(writer, sheet_name=f'{m} Premiums'[:31], index=False)
print(f"exported {file_name}")


## Next
`04_analysis.ipynb` builds the deep-dive: §1–§9 figures, validation, cross-scenario table, leak check, Monte Carlo, verdicts. Improvement backlog lives in `voltvision/simulate.py` + `voltvision/pricing.py` TODOs.